# Search complexity and observed agreement rates

The main analysis has **one observation per empirical study**: an experiment or
independently recruited sample. Independent studies using the same negotiation
scenario remain separate observations with their own agreement rates.

This notebook first rebuilds the study dataset from the recorded source outcomes
and structural scores. It then runs all six models and the supplementary checks.
The earlier analysis with one row per payoff structure and reservation-value
combination, and its stricter subset, are retained as robustness analyses.

Run the cells in order. No API key, account, paper download or private path is
required. The source papers and licensed teaching materials are not redistributed;
the notebook starts from the coding recorded in the included CSV files.

In [ ]:
from pathlib import Path
import json
import platform
import importlib.metadata
import numpy as np
import pandas as pd

BASE = Path.cwd()
OUTPUT = BASE/'outputs'
OUTPUT.mkdir(exist_ok=True)
DRAWS = 5000
studies = pd.read_csv(BASE/'data/study_level_dataset.csv')
structures = pd.read_csv(BASE/'data/sources/structures.csv')
outcomes = pd.read_csv(BASE/'data/sources/study_outcomes.csv')
outcomes['coding_note'] = outcomes.coding_note.fillna('')
versions = {name:importlib.metadata.version(name) for name in
            ['numpy','pandas','scipy','statsmodels','matplotlib']}
versions['python'] = platform.python_version()
(OUTPUT/'software_versions.json').write_text(json.dumps(versions,indent=2),encoding='utf-8')
print(versions)

## Rebuild one row per study

The source records contain each structure's previously calculated measures. Here we average those
measures within a study using the number of negotiation encounters assigned to
each structure. Agreement rate uses the same weights. When exact counts exist,
this is total agreements divided by total negotiations. A missing condition
denominator is not invented; equal weights are used only when no defensible
allocation is available, and that study has no negotiation-count weight.

Separate experiments or independently recruited samples remain separate. Conditions within one experiment,
or two tasks completed by the same sample, do not become independent studies.
The study-level measures are averages of separately calculated measures, not
measures calculated from an averaged payoff table or averaged reservation value.

Some older structural records already average known payoff possibilities. Their
recorded calculation is retained and explained in data/sources/coding_evidence.csv. The source
outcomes do not invent condition rates when only a study total is available.

In [ ]:
METRICS = ['bargaining_zone_share','search_complexity',
           'number_of_issues','number_of_parties','number_of_compatible_issues',
           'automated_counterpart','non_unanimous_rule']

def rebuild_studies(outcomes, structures):
    # Attach the archived structural scores to their separate study outcomes.
    columns = ['observation_id', *METRICS, 'payoff_source']
    joined = outcomes.merge(structures[columns], left_on='structure_id',
                            right_on='observation_id', how='left', validate='many_to_one')
    assert joined[METRICS].notna().all().all()
    records = []
    for study_id, group in joined.groupby('study_id', sort=False):
        complete = group.negotiations.notna().all() and group.negotiations.gt(0).all()
        weights = group.negotiations.to_numpy(float) if complete else np.ones(len(group))
        weights = weights / weights.sum()
        records.append({
            'observation_id':study_id, 'paper_citation':group.paper_citation.iloc[0],
            'study_label':group.study_label.iloc[0],
            'negotiation_scenario':' | '.join(group.negotiation_scenario.unique()),
            'agreement_rate':float(np.dot(weights,group.agreement_rate)),
            'agreements':group.agreements.sum() if complete and group.agreements.notna().all() else np.nan,
            'negotiations':group.negotiations.sum() if complete else np.nan,
            **{col:float(np.dot(weights,group[col])) for col in METRICS},
            'matched_structure_count':group.structure_id.nunique(),
            'source_component_count':len(group),
            'within_study_weighting':'Negotiation counts' if complete else 'Equal weights; condition counts unavailable',
            'source_components':' | '.join(group.component_id),
            'payoff_source':' | '.join(group.payoff_source.unique()),
            'agreement_rate_source':' | '.join(group.agreement_rate_source.unique()),
            'source_link':' | '.join(group.source_link.dropna().unique()),
            'coding_note':' | '.join(group.coding_note[group.coding_note.ne('')].unique()),
        })
    rebuilt = pd.DataFrame(records)
    # Supplementary checks allow errors to be related for studies sharing a
    # scenario. If one study uses two tasks, their scenario groups are connected.
    # This does not combine or drop studies in the primary analysis.
    parent = {name:name for name in joined.negotiation_scenario.unique()}
    def root(name):
        while parent[name] != name:
            parent[name] = parent[parent[name]]
            name = parent[name]
        return name
    for _, group in joined.groupby('study_id', sort=False):
        names = list(group.negotiation_scenario.unique())
        for name in names[1:]:
            parent[root(name)] = root(names[0])
    rebuilt['scenario_cluster'] = [root(group.negotiation_scenario.iloc[0])
                                    for _,group in joined.groupby('study_id',sort=False)]
    return rebuilt

rebuilt = rebuild_studies(outcomes, structures)
pd.testing.assert_frame_equal(rebuilt.fillna(''), studies.fillna(''),
                              check_dtype=False, check_exact=False, atol=1e-11, rtol=1e-11)
assert studies.observation_id.is_unique
print(f'Rebuilt all {len(rebuilt)} study observations from {len(outcomes)} outcome components.')


## Models and interpretation

- M1: search complexity.
- M2: search complexity and five basic controls.
- M3: search complexity, bargaining-zone share and the five controls.
- M4: bargaining-zone share.
- M5: bargaining-zone share and the same controls.
- M6: M3 weighted by the study's number of negotiation encounters.

The controls are issues, parties, compatible issues, an automated counterpart,
and a non-unanimous decision rule. A fraction describes a study combining
conditions. Continuous predictors are standardized within the rows used by each
model; agreement rate remains a proportion. Thus b = -.10 means ten percentage
points less agreement per one-SD increase. These are not fully standardized betas.

H1 uses the positive M4 bargaining-zone coefficient; H2 the negative M1 search
coefficient; H3 compares R²(M2) minus R²(M5); H4 uses the negative M3 search
coefficient. We apply the archived support rule: full-sample direction and
significance, a 5,000-draw observation-bootstrap interval in that direction, and
the same result after removing each observation in turn. H3 instead requires a
positive paired R² difference and interval, with a positive difference in every
omission. The study-level unit is a revision to the earlier recorded plan.

HC3 standard errors allow unequal residual variability. One study uses an
automated counterpart, so that dummy fits it exactly. Its coefficient is kept,
but its own HC3 uncertainty cannot be estimated. The code absorbs this dummy
when calculating the other coefficients' uncertainty and checks that their
point estimates are unchanged. The stricter structural data similarly contain
one non-unanimous-rule record and no automated counterparts.

In [ ]:
"""Readable analysis code inserted into the public replication notebook."""
from pathlib import Path
import json
import warnings
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

BASIC = ['number_of_issues', 'number_of_parties', 'number_of_compatible_issues',
         'automated_counterpart', 'non_unanimous_rule']
CONTINUOUS = ['search_complexity', 'bargaining_zone_share', *BASIC[:3]]
MODELS = {
    'M1': ['search_complexity'],
    'M2': ['search_complexity', *BASIC],
    'M3': ['search_complexity', 'bargaining_zone_share', *BASIC],
    'M4': ['bargaining_zone_share'],
    'M5': ['bargaining_zone_share', *BASIC],
    'M6': ['search_complexity', 'bargaining_zone_share', *BASIC],
}
TESTS = {'H1':('M4','bargaining_zone_share',1), 'H2':('M1','search_complexity',-1),
         'H4':('M3','search_complexity',-1)}

def scenario_group(sample):
    """Keep shared scenarios together in supplementary uncertainty checks."""
    return 'scenario_cluster' if 'scenario_cluster' in sample else 'negotiation_scenario'

def analysis_sample(data, tier):
    """Use the strict outcome evidence only for rows in the strict analysis."""
    sample = data.loc[data[f'{tier}_eligible']].copy()
    if tier == 'strict':
        for col in ['agreement_rate','agreements','negotiations']:
            sample[col] = sample[f'strict_{col}']
    return sample.reset_index(drop=True)

def validate_data(data):
    """Check units, identifiers and mathematical limits before analysis."""
    assert data.observation_id.is_unique
    assert not (data.strict_eligible & ~data.main_eligible).any()
    for col in ['agreement_rate','bargaining_zone_share','search_complexity']:
        assert data[col].between(0,1).all(), col
    assert (data.search_complexity <= 1-data.bargaining_zone_share+1e-9).all()
    assert (data.number_of_issues >= 2).all()
    assert (data.number_of_parties >= 2).all()
    assert data.loc[data.main_eligible, ['agreement_rate','search_complexity','bargaining_zone_share',*BASIC]].notna().all().all()
    assert data.loc[data.strict_eligible,'strict_agreement_rate'].between(0,1).all()
    # A mean of study rates can differ from the pooled numerator / denominator.
    # Therefore no check incorrectly forces every primary rate to equal A / N.
    for tier in ['main','strict']:
        sample = analysis_sample(data,tier)
        complete = sample[['agreements','negotiations']].dropna()
        assert (complete.agreements >= 0).all()
        assert (complete.agreements <= complete.negotiations).all()
    return {'retained_records':len(data), **{tier:{'observations':len(analysis_sample(data,tier)),
            'scenario_names':analysis_sample(data,tier).negotiation_scenario.nunique()} for tier in ['main','strict']}}

def fit(sample, model):
    """Estimate OLS, or negotiation-count WLS for M6, with HC3 uncertainty.

    We standardize continuous X variables *after* choosing the model's rows.
    Y stays a proportion. A coefficient of -.10 means 10 percentage points less
    agreement for a one-SD increase in X. These are not fully standardized betas.
    """
    predictors = MODELS[model]
    use = sample.dropna(subset=['agreement_rate',*predictors]).copy()
    if model == 'M6':
        use = use.loc[use.negotiations.notna() & use.negotiations.gt(0)]
    X = use[predictors].astype(float).copy()
    for col in predictors:
        if col in CONTINUOUS:
            sd = X[col].std(ddof=0)
            X[col] = (X[col]-X[col].mean())/sd if sd > 1e-12 else 0.
    X.insert(0,'const',1.)
    # Rare controls can be constant or redundant in a resample. Keep the focal
    # predictors first, then retain only controls adding independent information.
    kept, dropped = ['const'], []
    for col in predictors:
        if np.linalg.matrix_rank(X[[*kept,col]].to_numpy()) > len(kept):
            kept.append(col)
        elif col in ['search_complexity','bargaining_zone_share']:
            raise ValueError(f'{model}: focal predictor {col} is not identifiable')
        else:
            dropped.append(col)
    X = X[kept]
    regression = sm.WLS(use.agreement_rate,X,weights=use.negotiations) if model=='M6' else sm.OLS(use.agreement_rate,X)
    ordinary = regression.fit()
    # A binary control occurring in only one row fits that row exactly. HC3's
    # residual/(1-leverage) then becomes 0/0. The other slopes are exactly the
    # same after absorbing that dummy and its singleton row. Compute their HC3
    # covariance on the remaining rows, preserving the original X scaling and
    # degrees of freedom. Uncertainty for the singleton dummy is unavailable.
    singleton = [col for col in X.columns if col!='const' and
                 col not in ['search_complexity','bargaining_zone_share'] and
                 int(X[col].ne(0).sum())==1]
    if singleton:
        keep_rows = ~X[singleton].ne(0).any(axis=1)
        remaining = [col for col in X if col not in singleton]
        reduced = sm.WLS(use.loc[keep_rows,'agreement_rate'],X.loc[keep_rows,remaining],
                         weights=use.loc[keep_rows,'negotiations']) if model=='M6' else sm.OLS(
                         use.loc[keep_rows,'agreement_rate'],X.loc[keep_rows,remaining])
        reduced_fit = reduced.fit(cov_type='HC3',use_t=True)
        np.testing.assert_allclose(ordinary.params[remaining],reduced_fit.params,atol=1e-9)
        se = reduced_fit.bse.reindex(X.columns)
        pv = pd.Series(2*stats.t.sf(np.abs(ordinary.params/se),ordinary.df_resid),index=X.columns)
        critical = stats.t.ppf(.975,ordinary.df_resid)
        ci = pd.DataFrame({0:ordinary.params-critical*se,1:ordinary.params+critical*se})
        fitted = SimpleNamespace(params=ordinary.params,bse=se,pvalues=pv,conf_int=lambda:ci,
                                 nobs=ordinary.nobs,rsquared=ordinary.rsquared,
                                 rsquared_adj=ordinary.rsquared_adj,fittedvalues=ordinary.fittedvalues)
        dropped.extend(f'{col}: singleton dummy; coefficient retained, HC3 SE unavailable' for col in singleton)
    else:
        fitted = regression.fit(cov_type='HC3',use_t=True)
    focal = [col for col in X if col in ['search_complexity','bargaining_zone_share']]
    if not np.isfinite(fitted.bse[focal]).all():
        raise ValueError(f'{model}: HC3 inference for a focal predictor is not finite')
    terms = pd.DataFrame({'term':fitted.params.index,'b':fitted.params.values,
                          'se_hc3':fitted.bse.values,'p':fitted.pvalues.values,
                          'ci_low':fitted.conf_int().iloc[:,0].values,
                          'ci_high':fitted.conf_int().iloc[:,1].values})
    return fitted, terms, use, dropped

def all_models(sample, tier):
    coefficients, models = [], []
    for model in MODELS:
        fitted, terms, use, dropped = fit(sample,model)
        terms.insert(0,'model',model)
        terms.insert(0,'dataset',tier)
        coefficients.append(terms)
        models.append({'dataset':tier,'model':model,'n':len(use),
                       'scenario_names':use.negotiation_scenario.nunique(),
                       'r_squared':float(fitted.rsquared),
                       'adjusted_r_squared':float(fitted.rsquared_adj),
                       'dropped_controls':'; '.join(dropped),
                       'predictions_below_zero':int((fitted.fittedvalues<0).sum()),
                       'predictions_above_one':int((fitted.fittedvalues>1).sum())})
    return pd.concat(coefficients,ignore_index=True),pd.DataFrame(models)

def hypothesis_estimates(sample):
    out = {}
    for h,(model,term,direction) in TESTS.items():
        fitted,_,_,_ = fit(sample,model)
        out[h] = {'estimate':float(fitted.params[term]),'p':float(fitted.pvalues[term]),
                  'full_sample_pass':bool(direction*fitted.params[term]>0 and fitted.pvalues[term]<.05)}
    m2 = fit(sample,'M2')[0]
    m5 = fit(sample,'M5')[0]
    assert m2.nobs == m5.nobs
    delta = m2.rsquared-m5.rsquared
    out['H3'] = {'estimate':float(delta),'p':None,'full_sample_pass':bool(delta>0)}
    return out

def fast_estimates(sample, weighted=False):
    """Least squares point estimates for the resampling calculations.

    This is algebraically the same model as statsmodels. Robust standard errors
    are unnecessary inside the percentile bootstrap, which uses only estimates.
    """
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    y = sample.agreement_rate.to_numpy(float)
    raw = sample[['search_complexity','bargaining_zone_share',*BASIC]].to_numpy(float)
    X = raw.copy()
    for j in range(5):
        sd = raw[:,j].std(ddof=0)
        X[:,j] = (raw[:,j]-raw[:,j].mean())/sd if sd>1e-12 else 0.
    X = np.column_stack([np.ones(len(y)),X])
    weights = sample.negotiations.to_numpy(float) if weighted else np.ones(len(y))
    root = np.sqrt(weights)
    total = np.sum(weights*(y-np.average(y,weights=weights))**2)
    if total<=0:
        raise ValueError('No outcome variation in resample')
    indices = {'M1':[0,1],'M2':[0,1,3,4,5,6,7], 'M3':list(range(8)),
               'M4':[0,2],'M5':[0,2,3,4,5,6,7]}
    fitted = {}
    for model in (['M3'] if weighted else indices):
        cols = indices[model]
        matrix = X[:,cols]
        # Reduce redundant controls in the same fixed order as fit().
        keep = [0]
        for j in range(1,len(cols)):
            if np.linalg.matrix_rank(matrix[:,[*keep,j]]) > len(keep):
                keep.append(j)
            elif cols[j] in [1,2]:
                raise ValueError('Focal predictor not identified in resample')
        matrix = matrix[:,keep]
        coefficient = np.linalg.lstsq(matrix*root[:,None],y*root,rcond=None)[0]
        r2 = 1-np.sum(weights*(y-matrix@coefficient)**2)/total
        fitted[model] = (coefficient,r2)
    if weighted:
        return {'M6':float(fitted['M3'][0][1])}
    return {'H1':float(fitted['M4'][0][1]),'H2':float(fitted['M1'][0][1]),
            'H3':float(fitted['M2'][1]-fitted['M5'][1]),'H4':float(fitted['M3'][0][1])}

def bootstrap(sample, draws=5000, seed=20260921, whole_scenarios=False, weighted=False):
    """Resample observations, or scenario names for the supplementary check."""
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    rng = np.random.default_rng(seed)
    groups = [g for _,g in sample.groupby(scenario_group(sample),sort=False)]
    values, failures = [], []
    for draw in range(draws):
        if whole_scenarios:
            resample = pd.concat([groups[j] for j in rng.integers(0,len(groups),len(groups))],ignore_index=True)
        else:
            resample = sample.iloc[rng.integers(0,len(sample),len(sample))]
        try:
            values.append(fast_estimates(resample,weighted=weighted))
        except ValueError as error:
            failures.append({'draw':draw+1,'reason':str(error)})
    results = pd.DataFrame(values)
    summary = []
    for h in results:
        low,high = results[h].quantile([.025,.975])
        positive = h in ['H1','H3']
        summary.append({'test':h,'ci_low':low,'ci_high':high,
                        'draws_requested':draws,'draws_completed':len(results),
                        'invalid_draws':len(failures),
                        'passes_interval_rule':bool(low>0 if positive else high<0)})
    return pd.DataFrame(summary),pd.DataFrame(failures)

def omission_checks(sample, whole_scenarios=False, weighted=False):
    """Apply direction AND significance, as required in the preregistration."""
    if weighted:
        sample = sample.loc[sample.negotiations.notna() & sample.negotiations.gt(0)]
    key = scenario_group(sample) if whole_scenarios else 'observation_id'
    rows = []
    for omitted in sample[key].unique():
        use = sample.loc[sample[key].ne(omitted)]
        if weighted:
            fitted = fit(use,'M6')[0]
            rows.append({'omitted':omitted,'test':'M6','estimate':float(fitted.params['search_complexity']),
                         'p':float(fitted.pvalues['search_complexity']),
                         'passes':bool(fitted.params['search_complexity']<0 and fitted.pvalues['search_complexity']<.05)})
        else:
            for h,value in hypothesis_estimates(use).items():
                rows.append({'omitted':omitted,'test':h,'estimate':value['estimate'],
                             'p':value['p'],'passes':value['full_sample_pass']})
    return pd.DataFrame(rows)

def collapse_scenarios(sample):
    """Average structural versions into one row per named scenario, as specified."""
    return sample.groupby('negotiation_scenario',as_index=False)[
        ['agreement_rate','search_complexity','bargaining_zone_share',*BASIC]].mean()

def describe(sample):
    columns = ['agreement_rate','bargaining_zone_share','search_complexity',*BASIC]
    summary = sample[columns].agg(['count','mean','std','min','max']).T
    correlation = sample[columns].corr()
    p = pd.DataFrame(np.nan,index=columns,columns=columns)
    for left in columns:
        for right in columns:
            if sample[left].std()>0 and sample[right].std()>0:
                p.loc[left,right] = stats.pearsonr(sample[left],sample[right]).pvalue
    return summary,correlation,p


def all_models_for_collapsed(sample,tier):
    co,mo = [],[]
    for model in ['M1','M2','M3','M4','M5']:
        fitted,terms,use,dropped = fit(sample,model)
        terms.insert(0,'model',model); terms.insert(0,'dataset',tier)
        co.append(terms)
        mo.append({'dataset':tier,'model':model,'n':len(use),'r_squared':float(fitted.rsquared)})
    return pd.concat(co,ignore_index=True),pd.DataFrame(mo)


## Run the primary and robustness analyses

The source outcome is not statistically imputed. Negotiation counts are not
participant counts. M6 omits studies without a valid count and uses their existing
agreement rate. The current study dataset has one missing count.

The earlier structural aggregation gives each payoff and reservation-value
combination one row, averaging independent study rates. Its stricter subset
removes documented structural assumptions, reconstructions and pooled structures.
Those checks ask whether the result changes when repeated uses of one structure
are combined and uncertain structural matches are removed.

We also report complete-scenario resampling and omissions, scenario-clustered
uncertainty, and averages over named scenarios as supplementary checks. Studies
sharing a scenario can have related errors; connected groups preserve studies
that use more than one task. These checks change uncertainty or weighting, not
the main definition of a study observation. Rank correlations are exploratory.
Every check is reported even when it is inconclusive.

In [ ]:
"""Study observations are primary; structural aggregation is supplementary."""
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

def cluster_coefficients(sample,model):
    """Allow regression errors to be related for studies sharing a scenario.

    This is a supplementary CR1 cluster covariance with its finite-sample
    correction and a t reference using G-1 scenario groups. Point estimates are
    exactly those from the primary model; only uncertainty changes.
    """
    fitted,_,use,_=fit(sample,model)
    predictors=MODELS[model]
    X=use[predictors].astype(float).copy()
    for col in predictors:
        if col in CONTINUOUS:
            sd=X[col].std(ddof=0)
            X[col]=(X[col]-X[col].mean())/sd if sd>1e-12 else 0.
    X.insert(0,'const',1.)
    kept=['const']
    for col in predictors:
        if np.linalg.matrix_rank(X[[*kept,col]].to_numpy())>len(kept):
            kept.append(col)
    X=X[kept]
    groups=use[scenario_group(use)]
    regression=sm.WLS(use.agreement_rate,X,weights=use.negotiations) if model=='M6' else sm.OLS(use.agreement_rate,X)
    result=regression.fit(cov_type='cluster',cov_kwds={'groups':groups,
                            'use_correction':True,'df_correction':True},use_t=True)
    np.testing.assert_allclose(result.params,fitted.params,atol=1e-10)
    out=pd.DataFrame({'term':result.params.index,'b':result.params.values,
                      'se_cluster':result.bse.values,'p':result.pvalues.values,
                      'ci_low':result.conf_int().iloc[:,0].values,
                      'ci_high':result.conf_int().iloc[:,1].values,
                      'scenario_groups':groups.nunique()})
    # A control observed only once has no independent replication. Do not imply
    # that its own sandwich SE supplies a generalizable test of that control.
    singleton=[col for col in X if col!='const' and X[col].ne(0).sum()==1]
    out.loc[out.term.isin(singleton),['se_cluster','p','ci_low','ci_high']]=np.nan
    return out

def run_revision(studies,structures,output,draws=5000):
    output=Path(output)
    output.mkdir(exist_ok=True,parents=True)
    samples={'study':studies.copy(), 'structure':analysis_sample(structures,'main'),
             'strict_structure':analysis_sample(structures,'strict')}
    validate_data(structures)
    assert studies.observation_id.is_unique
    coefficients=[];models=[];bootstraps=[];omissions=[];supports=[];clusters=[];averages=[]
    started=time.perf_counter()
    for offset,(name,sample) in enumerate(samples.items()):
        print(f'{name}: {len(sample)} rows',flush=True)
        for model in MODELS:
            terms=cluster_coefficients(sample,model)
            terms.insert(0,'model',model);terms.insert(0,'dataset',name)
            clusters.append(terms)
        co,mo=all_models(sample,name)
        coefficients.append(co);models.append(mo)
        d,c,p=describe(sample)
        for label,frame in [('descriptive_statistics',d),('correlations',c),('correlation_p_values',p)]:
            frame.to_csv(output/f'{name}_{label}.csv')
        # Check the fast resampling calculations against ordinary statsmodels.
        slow=hypothesis_estimates(sample)
        for h,value in fast_estimates(sample).items():
            np.testing.assert_allclose(value,slow[h]['estimate'],atol=1e-11)
        np.testing.assert_allclose(fast_estimates(sample,True)['M6'],fit(sample,'M6')[0].params['search_complexity'],atol=1e-11)
        for whole in [False,True]:
            for weighted in [False,True]:
                kind=('scenario' if whole else 'observation')+('_weighted' if weighted else '')
                intervals,failures=bootstrap(sample,draws,20261105+offset+100*int(whole)+200*int(weighted),whole,weighted)
                intervals.insert(0,'resampling',kind);intervals.insert(0,'dataset',name)
                bootstraps.append(intervals)
                if len(failures):
                    failures.to_csv(output/f'{name}_{kind}_invalid_draws.csv',index=False)
                om=omission_checks(sample,whole,weighted)
                om.insert(0,'omission',kind);om.insert(0,'dataset',name)
                omissions.append(om)
                if not whole and not weighted:
                    for h,full in slow.items():
                        bs=intervals.loc[intervals.test.eq(h)].iloc[0]
                        leave=om.loc[om.test.eq(h)]
                        supports.append({'dataset':name,'hypothesis':h,**full,
                            'bootstrap_ci_low':bs.ci_low,'bootstrap_ci_high':bs.ci_high,
                            'bootstrap_pass':bool(bs.passes_interval_rule),
                            'omission_refits':len(leave),'omission_refits_passing':int(leave.passes.sum()),
                            'supported_by_specified_tests':bool(full['full_sample_pass'] and bs.passes_interval_rule and leave.passes.all())})
                print('  complete '+kind,flush=True)
        if name!='study':
            collapsed=collapse_scenarios(sample)
            co,mo=all_models_for_collapsed(collapsed,name)
            averages.append(co.merge(mo,on=['dataset','model']))
    for label,frames in [('coefficients',coefficients),('model_fit',models),
                         ('bootstrap_intervals',bootstraps),('omission_checks',omissions),
                         ('clustered_coefficients',clusters),('scenario_averaged_results',averages)]:
        pd.concat(frames,ignore_index=True).to_csv(output/f'{label}.csv',index=False)
    support=pd.DataFrame(supports).sort_values(['dataset','hypothesis'])
    support.to_csv(output/'hypothesis_tests.csv',index=False)
    # A short exploratory check uses ranks instead of distances on the scales.
    exploratory=[]
    for variable in ['bargaining_zone_share','search_complexity']:
        result=stats.spearmanr(studies[variable],studies.agreement_rate)
        exploratory.append({'predictor':variable,'spearman_rho':result.statistic,'p':result.pvalue})
    pd.DataFrame(exploratory).to_csv(output/'exploratory_rank_correlations.csv',index=False)
    summary={'study_observations':len(studies),'source_papers':studies.paper_citation.nunique(),
             'scenario_groups':studies.scenario_cluster.nunique(),
             'studies_at_100_percent_agreement':int(np.isclose(studies.agreement_rate,1).sum()),
             'structure_observations':len(samples['structure']),
             'strict_structure_observations':len(samples['strict_structure']),
             'bootstrap_draws_per_check':draws,'seconds':time.perf_counter()-started}
    (output/'analysis_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(support.to_string(index=False),flush=True)
    print(json.dumps(summary,indent=2),flush=True)
    return support


In [ ]:
support = run_revision(studies, structures, OUTPUT, draws=DRAWS)
support

## Read the results

The main study-level analysis supports H2-H4 under the stated regression,
resampling and omission tests. H1 is not supported. Search complexity remains
negative in the adjusted and negotiation-weighted models. The earlier structural
aggregation supports H1-H4; its stricter subset supports H2-H4.

This does not mean every supplementary check supports every prediction. The
exploratory rank correlations are not significant. Complete-scenario intervals
for the study-level H2 and H3 tests include zero, while the adjusted H4 interval
stays negative. See the generated CSV files for every coefficient, interval and
omission, rather than selecting only significant results.

The weighted R² uses a different weighting of variation, so its size is not a
direct improvement over the ordinary R². Five M3 fitted values exceed one. OLS
is used here to describe associations, not to produce calibrated probabilities.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams.update({'font.family':'Times New Roman','font.size':15,
    'axes.labelsize':17,'axes.titlesize':17,'xtick.labelsize':14,'ytick.labelsize':14,
    'legend.fontsize':14,'svg.fonttype':'none'})

fig, axes = plt.subplots(1,2,figsize=(10,4.7),constrained_layout=True)
for ax, variable, label in zip(axes,['bargaining_zone_share','search_complexity'],
                                 ['Bargaining-zone share','Search complexity']):
    ax.scatter(studies[variable],studies.agreement_rate,s=38,color='#555555',
               alpha=.50,edgecolors='white',linewidths=.35)
    slope, intercept = np.polyfit(studies[variable],studies.agreement_rate,1)
    x = np.array([studies[variable].min(),studies[variable].max()])
    ax.plot(x,intercept+slope*x,color='black',linewidth=1.5)
    ax.set(xlabel=label,ylabel='Observed agreement rate',xlim=(-.03,1.03),ylim=(-.03,1.03))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(False)
for extension in ['png','svg']:
    fig.savefig(OUTPUT/f'figure_1_study_associations.{extension}',dpi=300,bbox_inches='tight')
plt.close(fig)

co = pd.read_csv(OUTPUT/'coefficients.csv')
fig, ax = plt.subplots(figsize=(9.6,4.8),constrained_layout=True)
labels = ['M1  Search complexity', 'M2  + basic controls',
          'M3  + bargaining-zone share', 'M6  Weighted by negotiations']
for model,y in zip(['M1','M2','M3','M6'],[3,2,1,0]):
    row = co.loc[co.dataset.eq('study') & co.model.eq(model) & co.term.eq('search_complexity')].iloc[0]
    ax.errorbar(row.b*100,y,xerr=[[(row.b-row.ci_low)*100],[(row.ci_high-row.b)*100]],
                fmt='o',color='black',markersize=8,capsize=5,linewidth=1.7)
ax.axvline(0,color='#999999',linewidth=1)
ax.set_yticks([3,2,1,0],labels)
ax.set_ylim(-.5,3.5)
ax.set_xlabel('Agreement-rate difference per one-SD increase'+chr(10)+'(percentage points)')
ax.spines[['top','right','left']].set_visible(False)
ax.grid(False)
for extension in ['png','svg']:
    fig.savefig(OUTPUT/f'figure_2_study_coefficients.{extension}',dpi=300,bbox_inches='tight')
plt.close(fig)
print('Saved the figures as PNG and SVG in outputs/.')
